In [ ]:
# Cell 1 — Setup
"""
02_train.ipynb
==============
Train TGNN-Solv with the three-phase curriculum.

CLI equivalents:
- `python scripts/training/train.py ...` for one run
- `python scripts/experiments/run_seeds.py ...` for multi-seed evaluation
- `python scripts/experiments/run_seeds.py --config configs/paper_config_split_late.yaml ...`
  for the matched split-late backbone comparison
- `python scripts/experiments/run_split_comparisons.py ...` for split-wise fair comparison
- `python scripts/training/train_directgnn.py ...` for the no-physics baseline
- optional Stage 0 pretraining via `scripts/training/train.py --pretrain`,
  `scripts/training/train_with_pretrain.py`, or `tgnn_solv.pretrain.Pretrainer`
- `bash reproduce.sh` for the full paper pipeline
"""

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
NOTEBOOK_FIG_DIR = FIGURES_DIR / "notebooks"
NOTEBOOK_RESULTS_DIR = RESULTS_DIR / "notebooks"
NOTEBOOK_FIG_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import torch
import pandas as pd

from tgnn_solv.config import TGNNSolvConfig
from tgnn_solv.model import TGNNSolv
from tgnn_solv.trainer import TGNNSolvTrainer
from tgnn_solv.inference import save_model
from tgnn_solv.data import make_loaders, PROCESSED_DIR

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {DEVICE}")


## Mathematical structure of TGNN-Solv and the curriculum

For each system the model sees two graphs and a temperature:

$$
\left(G^{\mathrm{sol}},\; G^{\mathrm{slv}},\; T\right) \longmapsto \widehat y, \qquad y = \ln x_2.
$$

The physics branch first builds the solver-based prediction

$$
\ln x_2^{\mathrm{phys}} = -\Phi\!\left(T, T_m, \Delta H_{\mathrm{fus}}, \Delta C_p\right)
- \ln \gamma_2\!\left(T, \tau_{12}, \tau_{21}, \alpha_{12}\right).
$$

The correction branch then proposes a corrected solver solution, but the residual is explicitly bounded:

$$
\ln x_2^{\mathrm{proposal}} = f_{\mathrm{SLE}}\!\left(T, \widetilde\theta_{\mathrm{fus}}, \widetilde\theta_{\mathrm{NRTL}}\right),
$$

$$
r = \operatorname{clip}\!\left(
\ln x_2^{\mathrm{proposal}} - \ln x_2^{\mathrm{phys}},
-r_{\max},
r_{\max}
\right),
$$

$$
\ln x_2^{\mathrm{final}} = \ln x_2^{\mathrm{phys}} + (1-c)\,r,
\qquad c \in [0,1].
$$

In the current implementation, the main supervised solubility term is a Huber loss:

$$
L_{\mathrm{sol}} = \frac{1}{M} \sum_{i=1}^{M} h_\delta\!\left(\widehat y_i - y_i\right),
$$

$$
h_\delta(e) =
\begin{cases}
\frac12 e^2, & |e| \le \delta, \\
\delta\left(|e| - \frac12\delta\right), & |e| > \delta.
\end{cases}
$$

The full objective in each phase is a weighted sum of components:

$$
L^{(p)} = \sum_k \lambda_k^{(p)} L_k,
$$

where the phase-dependent weights change for `sol`, `T_m`, `dH`, `bridge`, `pair_temp_rank`, `vant_hoff_local`, `walden`, and other regularizers.


## Step 1. Load the canonical split and define the budget

The practical part starts here: scaffold split, model configuration, and
training budget. At this point it is useful to line up the chosen epoch
budget, batch size, encoder family (`mpnn` vs `gps`), descriptor augmentation,
and optional priors with the architectural hypothesis you actually want to test.


In [ ]:
# Cell 2 — Load processed data
# Use the canonical scaffold split by default.
# For comparisons against prior work that used random-by-solute splits,
# swap in `train_solute.csv` / `val_solute.csv` / `test_solute.csv`.
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

print(f"Train: {len(train_df):,}")
print(f"Val:   {len(val_df):,}")
print(f"Test:  {len(test_df):,}")


In [ ]:
# Cell 3 — Configuration
cfg = TGNNSolvConfig(
    hidden_dim=128,
    n_gnn_layers=4,
    encoder_role_mode="shared_residual",
    encoder_role_specific_layers=1,
    n_cross_attn_layers=1,
    n_attn_heads=4,
    pair_dim=256,
    dropout=0.35,
    nrtl_tau_mode="ref_invT",
    use_temperature_in_encoder=False,
    use_temperature_in_interaction=False,
    use_temperature_in_nrtl_head=True,
    n_iter_train=3,
    n_iter_eval=10,
    use_implicit_diff=True,
    use_pair_temperature_batching=True,
    pair_temperature_min_group_size=2,
    pair_temperature_group_chunk_size=4,
    batch_size=128,
    lr_phase1=3e-4,
    lr_phase2=3e-4,
    lr_phase3=3e-5,
    epochs_phase1=2,
    epochs_phase2=5,
    epochs_phase3=1,
    warmup_epochs=5,
    patience=40,
    S_g=5000.0,
    tau_clamp=15.0,
)

print(f"Config: hidden={cfg.hidden_dim}, layers={cfg.n_gnn_layers}")
print(f"Encoder mode: {cfg.encoder_role_mode}")
print(f"NRTL mode: {cfg.nrtl_tau_mode}")


## Step 2. Check model size and the physical structure

After defining `TGNNSolvConfig`, the notebook instantiates a concrete model.
This is the moment to verify that the parameter count and the presence of the
solver path match expectations before spending time on a full run.


In [ ]:
# Cell 4 — Build model
model = TGNNSolv(cfg=cfg).to(DEVICE)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
physics = sum(p.numel() for p in model.sle_solver.parameters())

print(f"Total params:     {total:,}")
print(f"Trainable params: {trainable:,}")
print(f"Physics params:   {physics} (should be 0)")

In [ ]:
# Cell 5 — DataLoaders
train_loader, val_loader, test_loader = make_loaders(
    train_df,
    val_df,
    test_df,
    batch_size=cfg.batch_size,
    use_pair_temperature_batching=cfg.use_pair_temperature_batching,
    pair_temperature_min_group_size=cfg.pair_temperature_min_group_size,
    pair_temperature_group_chunk_size=cfg.pair_temperature_group_chunk_size,
)

print(f"Pair-temperature batching: {cfg.use_pair_temperature_batching}")


## Stage 0 pretraining formulas

The separate `Pretrainer` from `pretrain.py` is still available directly,
but the maintained CLI also supports the same Stage 0 through
`train.py --pretrain` or `--pretrain-checkpoint`: it is an optional Stage 0 that updates
`model.gnn` and `model.readout` before the three-phase curriculum starts.

Its multi-task objective is

$$
L_{\mathrm{stage0}} =
\lambda_{\mathrm{atom}} L_{\mathrm{atom}} +
\lambda_{\mathrm{bond}} L_{\mathrm{bond}} +
\lambda_{\mathrm{prop}} L_{\mathrm{prop}} +
\lambda_{\mathrm{ctr}} L_{\mathrm{ctr}}.
$$

Here

$$
L_{\mathrm{atom}} = \frac{1}{|\mathcal{M}|} \sum_{v \in \mathcal{M}}
\left\| \widehat{x}_v - x_v \right\|_2^2
$$

reconstructs masked subgraph atom features,

$$
L_{\mathrm{bond}} = -\frac{1}{|\mathcal{E}_m|}
\sum_{e \in \mathcal{E}_m} \log p_\theta\!\left(b_e \mid h_{\mathrm{src}(e)}, h_{\mathrm{dst}(e)}\right)
$$

classifies masked bond types,

$$
L_{\mathrm{prop}} = \frac{1}{B} \sum_{i=1}^{B}
\left\| \widehat{p}_i - p_i^{\mathrm{RDKit}} \right\|_2^2
$$

predicts cheap RDKit properties,
and the contrastive term uses two augmented views of the same molecule:

$$
L_{\mathrm{ctr}} = \frac12
\Bigl[
\mathrm{CE}\!\left( z_1 z_2^\top / \tau, I \right) +
\mathrm{CE}\!\left( z_2 z_1^\top / \tau, I \right)
\Bigr].
$$

In practice this gives an encoder initialization that is richer in chemical
invariants than a purely random start.


## Step 3. Optional Stage 0 before the curriculum

You can skip this block if you want to reproduce the standard CLI path without Stage 0.
If you do enable Stage 0, it is best interpreted as a separate
encoder/readout initialization step, not as part of Phase 1 in the main trainer.
The same behavior is available from the maintained CLI through
`scripts/training/train.py --pretrain` and `scripts/training/train_with_pretrain.py`.


In [ ]:
# Cell 5b — Pretrain GNN (Stage 0)
from tgnn_solv.pretrain import Pretrainer, download_zinc250k

# Optional Stage 0. This happens before the trainer's Phase 1/2/3 curriculum.
# The pretrainer updates `model.gnn` and `model.readout` in place, then
# discards its temporary auxiliary heads.
# The maintained CLI can drive the same path with `--pretrain` or
# `--pretrain-checkpoint`.
# Download or collect SMILES
smiles_for_pretrain = download_zinc250k()

# Pretrain GNN encoder + readout
pretrainer = Pretrainer(model.gnn, model.readout, cfg, DEVICE)
pretrain_history = pretrainer.pretrain(
    smiles_for_pretrain,
    n_epochs=2,          # 30 epochs
    batch_size=128,
    lr=3e-4,
    mask_ratio=0.15,
    mask_hops=2,
    bond_mask_ratio=0.15,
    aug_node_mask_ratio=0.15,
    aug_edge_mask_ratio=0.15,
    bond_loss_weight=0.5,
    contrastive_weight=0.5,
    contrastive_temp=0.1,
)

# Visualize pretraining loss
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(pretrain_history["atom"], label="Masked subgraph", color="steelblue")
ax.plot(pretrain_history["bond"], label="Bond pred", color="slateblue")
ax.plot(pretrain_history["prop"], label="Property pred", color="coral")
ax.plot(pretrain_history["contrastive"], label="Contrastive", color="seagreen")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Stage 0: GNN Pretraining")
ax.legend()
plt.tight_layout()
plt.show()

print("GNN pretrained. Proceeding to curriculum training...")

In [ ]:
# Cell 6 — Forward pass smoke test
model.eval()
for sol_b, slv_b, tgt in train_loader:
    sol_b = sol_b.to(DEVICE)
    slv_b = slv_b.to(DEVICE)
    T = tgt["T"].to(DEVICE)
    with torch.no_grad():
        out = model(sol_b, slv_b, T)
    print("Forward pass OK")
    print(f"  ln(x₂) shape: {out['ln_x2'].shape}")
    print(f"  T_m range: [{out['fusion_params']['T_m'].min():.0f}, "
          f"{out['fusion_params']['T_m'].max():.0f}]")
    print(f"  Gate: {out['gate'].item():.4f}")
    break

## Temperature regularizers used during training

The second important part of the curriculum is regularization over the
temperature structure inside the same `(solute, solvent)` pair.

Monotonicity is implemented through the derivative with respect to temperature:

$$
L_{\mathrm{mono}} = \frac{1}{B} \sum_{i=1}^{B}
\operatorname{ReLU}\!\left(-\frac{\partial \widehat y_i}{\partial T_i}\right).
$$

For same-pair ranking, after sorting by temperature
\(T_{(1)} < T_{(2)} < \dots < T_{(m)}\), the model is penalized when `ln(x_2)` decreases:

$$
L_{\mathrm{rank}} = \frac{1}{m-1}
\sum_{k=1}^{m-1}
\operatorname{ReLU}\!\left(-\bigl(\widehat y_{(k+1)} - \widehat y_{(k)}\bigr)\right).
$$

The local van't Hoff regularizer works with slopes in the
\(1/T\) coordinate:

$$
\Delta_k = T_{(k+1)}^{-1} - T_{(k)}^{-1},
$$

$$
\Delta_k^{\mathrm{safe}} = \operatorname{sign}(\Delta_k)
\max\!\left(|\Delta_k|, 10^{-4}\right),
$$

$$
s_k = \frac{\widehat y_{(k+1)} - \widehat y_{(k)}}{\Delta_k^{\mathrm{safe}}}.
$$

The change between neighboring local slopes is then penalized:

$$
L_{\mathrm{vH}} = \operatorname{mean}
\left[
\operatorname{clip}\!\left((s_{k+1} - s_k)^2,\; 0,\; 100\right)
\right].
$$

This is exactly why pair-aware batching matters: without multiple temperatures
for the same pair, these terms do not see enough structure to learn from.


## Step 4. Launch the three-phase curriculum

This is where training actually begins: Phase 1 performs auxiliary warmup,
Phase 2 activates the solubility objective and the correction path, and
Phase 3 is a low-LR refinement stage. The learning curves only make sense if
you read them with these phase transitions in mind.


In [ ]:
# Cell 7 — Train
trainer = TGNNSolvTrainer(model, cfg)
trainer.train_full(train_loader, val_loader)

## Step 5. Read the training curves

In the plots below, it is useful to watch not only the total loss, but also
when validation MAE begins to improve and how the correction gate behaves.
Sharp jumps at the beginning of Phase 2 usually mean the regularizer weights
deserve especially careful reading.


In [ ]:
# Cell 8 — Training curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Loss
axes[0].plot(trainer.history["train_loss"], label="train", alpha=0.7)
axes[0].plot(trainer.history["val_loss"], label="val", alpha=0.7)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_yscale("log")
axes[0].legend()
axes[0].set_title("Loss")

# Phase boundaries
phases = trainer.history["phase"]
for p in [1, 2, 3]:
    starts = [i for i, x in enumerate(phases) if x == p]
    if starts:
        for ax in axes:
            ax.axvline(starts[0], color="gray", ls=":", alpha=0.5)

# MAE
if trainer.history["val_mae"]:
    axes[1].plot(trainer.history["val_mae"], color="green")
    axes[1].set_xlabel("Step (Phase 2+)")
    axes[1].set_ylabel("MAE (ln x_2)")
    axes[1].set_title("Validation MAE")

# Gate
axes[2].plot(
    [torch.tanh(torch.tensor(g)).item() for g in trainer.history["gate"]],
    color="orange",
)
axes[2].set_xlabel("Step")
axes[2].set_ylabel("tanh(w_0)")
axes[2].set_title("Correction gate")
axes[2].set_ylim(-0.1, 1.0)

plt.tight_layout()
plt.savefig(NOTEBOOK_FIG_DIR / "training_curves.png", dpi=150)
plt.show()


## Step 6. Final check on the test split

After training, the notebook performs the minimum viable regression check:
MAE, RMSE, R², and a parity plot. That is not enough for a paper-grade
comparison, but as an interactive sanity check it is fast and informative.


In [ ]:
# Cell 9 — Evaluate on test set
# For a reproducible JSON report, use: python scripts/evaluation/evaluate_complete.py ...
from tgnn_solv.loss import TGNNSolvLoss

model.eval()
loss_fn = TGNNSolvLoss(cfg)
all_pred, all_true = [], []

with torch.no_grad():
    for sol_b, slv_b, tgt in test_loader:
        sol_b = sol_b.to(DEVICE)
        slv_b = slv_b.to(DEVICE)
        T = tgt["T"].to(DEVICE)
        mask = tgt["has_solubility"].to(DEVICE)

        out = model(sol_b, slv_b, T)
        if mask.any():
            all_pred.append(out["ln_x2"][mask].cpu())
            all_true.append(tgt["ln_x2"][mask].cpu())

pred = torch.cat(all_pred)
true = torch.cat(all_true)

mae = (pred - true).abs().mean().item()
rmse = (pred - true).pow(2).mean().sqrt().item()
r2_num = (pred - true).pow(2).sum()
r2_den = (true - true.mean()).pow(2).sum()
r2 = 1.0 - (r2_num / (r2_den + 1e-8)).item()

print(f"Test set results (n={len(pred):,}):")
print(f"  MAE  = {mae:.3f} ln-units")
print(f"  RMSE = {rmse:.3f} ln-units")
print(f"  R²   = {r2:.4f}")

# Parity plot
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(true.numpy(), pred.numpy(), s=4, alpha=0.3)
lims = [min(true.min(), pred.min()) - 1, max(true.max(), pred.max()) + 1]
ax.plot(lims, lims, "r--", lw=1)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Experimental ln(x_2)")
ax.set_ylabel("Predicted ln(x_2)")
ax.set_title(f"Test set: MAE={mae:.3f}, R²={r2:.3f}")
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(NOTEBOOK_FIG_DIR / "parity_plot.png", dpi=150)
plt.show()


## Step 7. Save the checkpoint and basic artifacts

The final step saves the model together with the core experiment metadata.
This is useful even in notebook-driven work, because the next inference or
evaluation notebook can then operate on an explicit checkpoint rather than a
live in-memory object.


In [ ]:
# Cell 10 — Save model
MODEL_PATH = CHECKPOINT_DIR / "tgnn_solv_trained.pt"
save_model(
    model, cfg,
    str(MODEL_PATH),
    metadata={
        "test_mae": mae,
        "test_rmse": rmse,
        "test_r2": r2,
        "n_train": len(train_df),
        "n_test": len(test_df),
    },
)
print(f"Saved checkpoint to {MODEL_PATH}")
print("For the matched backbone comparison, repeat the sweep with configs/paper_config_split_late.yaml.")
